# Colab training runner

Runs this project's training/tuning entry points (`src/models/{lstm,gcn,tune}.py`) on a Colab GPU (free-tier T4), because local MPS training is too slow/hot for full sweeps. See `local-notes/log.md` (2026-07-10/11 entries) for why this pipeline exists and the bugs it already survived.

**Colab has no headless/API-driven execution** -- this notebook is run manually, cell by cell, by a human sitting in a Colab session. Nothing here is triggered automatically from the local machine.

## Workflow

1. **Local machine** (outside this notebook): build `bundle.zip` and upload it, alongside this notebook, to Google Drive at `My Drive/Project/GNN Citi Bike Forecast/colab/` via `rclone`. The bundle must contain:
   - This repo's code (`src/`, `pyproject.toml`, etc.)
   - A restricted `data/processed/` (only what training needs -- excluding large raw/EDA-only artifacts)
   - `mlflow.db` + `mlruns/` (to keep run lineage continuous across machines)
   - `optuna.db`, **if resuming a prior sweep** -- Phase 7's tuner persists study state here so a multi-hour sweep survives a Colab disconnect; omit it only when starting a brand-new study
   - `GIT_PROVENANCE.json`, generated locally first via `python scripts/write_git_provenance.py` (this bundle ships code without a real `.git` checkout, so `mlflow_utils.get_git_sha()` falls back to this file)
2. **This notebook**: mount Drive, extract the bundle, install the few extra dependencies Colab doesn't ship, then run whichever section(s) below match this session's goal.
3. **This notebook**, after each stage: push results back to `colab/results/` on Drive (`mlflow.db`, `mlruns/`, `models/`, `optuna.db`) as a timestamped zip, and write a `DONE.txt` marker at the very end so a local poller watching Drive knows the session finished.
4. **Local machine**: pull the results back via `rclone`, merge into local `mlflow.db`/`mlruns`/`models`/`optuna.db`.

## Known pitfall this notebook already guards against

Shipping `mlflow.db` wholesale from the Mac to Colab once caused every artifact write (`mlflow.log_artifact`) to silently land on a bogus path baked into the DB from the *original* creating machine (`src/models/mlflow_utils.py::ensure_portable_artifact_location`, added after that incident). It self-heals automatically as long as this notebook's cwd is the repo root **and** `mlflow.db` lives there too (Setup section below enforces both) -- every training/tuning entry point calls it before `mlflow.set_experiment()`. Don't skip the Setup section.

## Sections are independent

Run only what this session needs, in order, after Setup:
- **Section A -- Full training run**: retrains all 9 model configs (baselines + LSTM + 6 GCN variants), the ROADMAP Phase 6 pattern.
- **Section B -- Hyperparameter tuning sweeps**: ROADMAP Phase 7's Optuna sweeps for LSTM and/or GCN.

You don't need to run both in the same session.

## Setup

Run every cell in this section, every session, regardless of which of Section A/B you're doing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

DRIVE_COLAB_DIR = Path("/content/drive/MyDrive/Project/GNN Citi Bike Forecast/colab")
DRIVE_RESULTS_DIR = DRIVE_COLAB_DIR / "results"
REPO_DIR = Path("/content/repo")

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Extract the most recently uploaded bundle -- `bundle.zip` if present,
# else the newest `bundle_*.zip` (hot-patch re-uploads use a suffixed name
# so a live session can pull a single fixed file without a full re-extract).
candidates = sorted(
    DRIVE_COLAB_DIR.glob("bundle*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError(f"No bundle*.zip found under {DRIVE_COLAB_DIR}")
bundle_path = candidates[0]
print(f"Extracting {bundle_path.name} -> {REPO_DIR}")

REPO_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle_path) as zf:
    zf.extractall(REPO_DIR)

In [ ]:
import sys

# cwd == mlflow.db's own directory, by convention (matches the local
# `sqlite:///mlflow.db` relative-URI usage) -- ensure_portable_artifact_
# location() anchors its "expected artifact path" fix to this directory,
# so keeping cwd and mlflow.db co-located here is what makes that fix work
# on Colab, not just locally.
os.chdir(REPO_DIR)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{REPO_DIR / 'mlflow.db'}"

sys.path.insert(0, str(REPO_DIR / "src" / "models"))
print("cwd:", Path.cwd())
print("MLFLOW_TRACKING_URI:", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
# Deliberately does NOT reinstall torch/torchvision/torchaudio -- Colab
# ships a CUDA-matched torch build already; replacing it risks a CUDA/torch
# version mismatch. Only install what's missing on top of that.
%pip install -q mlflow optuna torch-geometric pyarrow fastparquet scikit-learn holidays python-dotenv pyyaml

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

data_dir = REPO_DIR / "data" / "processed"
required = [
    "features_train.parquet", "features_val.parquet", "features_test.parquet",
    "targets_train.parquet", "targets_val.parquet", "targets_test.parquet",
    "graph_data.pkl", "feature_metadata.json", "SNAPSHOT.json",
]
missing = [f for f in required if not (data_dir / f).exists()]
if missing:
    raise FileNotFoundError(f"Bundle is missing expected data/processed/ files: {missing}")
print("data/processed/ looks complete.")

print("mlflow.db present:", (REPO_DIR / "mlflow.db").exists())
print("optuna.db present:", (REPO_DIR / "optuna.db").exists(), "(fine if absent -- a fresh study will be created)")

In [ ]:
def push_results_to_drive(tag: str) -> Path:
    """Zip mlflow.db/mlruns/models/optuna.db and upload to Drive's results/
    folder as a timestamped file, so a local rclone poller can pull it back.
    Safe to call repeatedly through a session (e.g. after each stage) --
    each call is a new, distinctly-named zip, nothing is overwritten.
    """
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    zip_name = f"results_{tag}_{timestamp}.zip"
    zip_path = Path("/content") / zip_name

    paths_to_zip = [
        REPO_DIR / "mlflow.db",
        REPO_DIR / "mlruns",
        REPO_DIR / "models",
        REPO_DIR / "optuna.db",
    ]
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in paths_to_zip:
            if not path.exists():
                continue
            if path.is_file():
                zf.write(path, arcname=path.name)
            else:
                for f in path.rglob("*"):
                    if f.is_file():
                        zf.write(f, arcname=str(f.relative_to(REPO_DIR)))

    dest = DRIVE_RESULTS_DIR / zip_name
    shutil.copy(zip_path, dest)
    print(f"Pushed {zip_path.stat().st_size / 1e6:.1f}MB -> {dest}")
    return dest


def mark_done() -> None:
    """Write the completion marker a local poller watches for."""
    (DRIVE_RESULTS_DIR / "DONE.txt").write_text(
        datetime.now(timezone.utc).isoformat() + "\n"
    )
    print("Wrote DONE.txt -- local poller should pick this up.")

## Section A -- Full training run (ROADMAP Phase 6 pattern)

Baselines, then LSTM, then all 6 GCN configs. Skip this section entirely if this session is only a Phase 7 tuning sweep (Section B).

In [ ]:
!python src/models/baseline.py --data-dir data/processed --model all

In [ ]:
!python src/models/lstm.py --data-dir data/processed

In [ ]:
push_results_to_drive("after_lstm")

In [ ]:
!python src/models/gcn.py --data-dir data/processed --adj all --loss-fn poisson

In [ ]:
!python src/models/gcn.py --data-dir data/processed --adj all --loss-fn mse

In [ ]:
push_results_to_drive("after_gcn")

## Section B -- Hyperparameter tuning sweeps (ROADMAP Phase 7)

Each `tune.py` invocation resumes its Optuna study from `optuna.db` if the bundle included one (`load_if_exists=True`), so re-running the same command after a disconnect continues rather than restarting from trial 0. Adjust `--n-trials`/`--timeout` for the session's actual time budget -- these are examples, not requirements.

In [ ]:
!python src/models/tune.py --model lstm --n-trials 30 --timeout 6h --data-dir data/processed

In [ ]:
push_results_to_drive("after_tune_lstm")

In [ ]:
!python src/models/tune.py --model gcn --n-trials 30 --timeout 6h --data-dir data/processed

In [ ]:
push_results_to_drive("after_tune_gcn")

## Section C -- Phase 8 spatiotemporal hybrid (BikeDemandSTGNN)

Six configs total: `{knn, flow, combined} x {poisson, mse}`.
`--adj all` loops over the three adjacency variants internally, mirroring
`gcn.py`. Each variant + loss combination lands as its own top-level MLflow
run.

Memory: default `--batch-size 4` is sized for T4 VRAM (see ROADMAP Phase 8).
If a run OOMs on `flow`/`combined` (denser edges), pass `--batch-size 2`.


In [ ]:
!python src/models/hybrid.py --data-dir data/processed --adj all --loss-fn poisson


In [ ]:
!python src/models/hybrid.py --data-dir data/processed --adj all --loss-fn mse


In [ ]:
push_results_to_drive("after_hybrid")


## Finish

Run this once, at the very end of the session, after whichever of Section A/B you ran.

In [ ]:
push_results_to_drive("FINAL")
mark_done()